In [ ]:
!pip install yfinance pandas tqdm pytz pycountry_convert dash

In [ ]:
# In fetch_index_data.py

import pandas as pd
import yfinance as yf
from tqdm import tqdm
import logging
import time
import pytz
import pycountry_convert as pc

# --- (Configuration and get_location_from_info function are unchanged) ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
INPUT_CSV = "/content/index_ticker_list_final.csv"
OUTPUT_CSV = "enriched_index_data.csv"

TIMEZONE_TO_COUNTRY_CODE = {}
for country_code, timezones in pytz.country_timezones.items():
    for timezone in timezones:
        TIMEZONE_TO_COUNTRY_CODE[timezone] = country_code

def get_location_from_info(info):
    timezone_str = info.get("exchangeTimezoneName")
    if timezone_str in TIMEZONE_TO_COUNTRY_CODE:
        try:
            country_code = TIMEZONE_TO_COUNTRY_CODE[timezone_str]
            continent_code = pc.country_alpha2_to_continent_code(country_code)
            return {
                "Country": pc.country_alpha2_to_country_name(country_code),
                "Continent": pc.convert_continent_code_to_continent_name(continent_code)
            }
        except Exception:
            pass
    return {"Country": "Global", "Continent": "Global"}

def enrich_index_data(input_file):
    # ... (file reading part is unchanged) ...
    try:
        df = pd.read_csv(input_file)
        if not all(col in df.columns for col in ["Index Name", "Yahoo Finance Ticker"]):
            logging.error("CSV must contain 'Index Name' and 'Yahoo Finance Ticker' columns.")
            return None
    except FileNotFoundError:
        logging.error(f"Input file not found: {input_file}. Please create it.")
        return None

    enriched_data = []

    for _, row in tqdm(df.iterrows(), total=df.shape[0], desc="Enriching Index Data"):
        original_index_name = row["Index Name"]
        ticker_str = row["Yahoo Finance Ticker"]

        try:
            ticker = yf.Ticker(ticker_str)
            info = ticker.info

            location = get_location_from_info(info)

            # <<< THE FIX FOR DATA LABELS IS HERE >>>
            # Create a fallback chain to find the best possible name.
            best_name = info.get("longName") or info.get("shortName") or original_index_name

            enriched_data.append({
                "Index Name": best_name, # Use the best name we found
                "Ticker": ticker_str,
                "Exchange": info.get("exchange", "N/A"),
                "Continent": location["Continent"],
                "Country": location["Country"],
            })
            time.sleep(0.1)
        except Exception as e:
            logging.warning(f"Could not fetch data for {ticker_str}: {e}. Skipping.")

    return pd.DataFrame(enriched_data)

# --- (__main__ block is unchanged) ---
if __name__ == "__main__":
    logging.info("Starting index data enrichment process...")
    enriched_df = enrich_index_data(INPUT_CSV)

    if enriched_df is not None and not enriched_df.empty:
        enriched_df['count'] = 1
        enriched_df.to_csv(OUTPUT_CSV, index=False)
        logging.info(f"Successfully saved enriched data to {OUTPUT_CSV}")
    else:
        logging.error("No data was enriched. Output file not created.")

In [ ]:
import pandas as pd
import plotly.express as px
import dash
from dash import dcc, html
from dash.dependencies import Input, Output

# --- (Data loading and App setup are unchanged) ---
try:
    df = pd.read_csv("enriched_index_data.csv")
except FileNotFoundError:
    print("FATAL ERROR: 'enriched_index_data.csv' not found. Please run the script first.")
    exit()

all_continents = sorted(df['Continent'].unique())
app = dash.Dash(__name__)
server = app.server
app.layout = html.Div([
    html.H1("Global Stock Index Dashboard", style={'textAlign': 'center'}),
    html.Div([
        html.Label("Select Continent(s):"),
        dcc.Dropdown(
            id='continent-filter',
            options=[{'label': i, 'value': i} for i in all_continents],
            value=all_continents, # Select all by default
            multi=True
        )
    ], style={'width': '80%', 'margin': 'auto', 'padding': '20px'}),
    dcc.Graph(id='sunburst-chart')
])

@app.callback(
    Output('sunburst-chart', 'figure'),
    Input('continent-filter', 'value')
)
def update_sunburst(selected_continents):
    if not selected_continents:
        return px.sunburst(title="Please select a continent to display data.")

    filtered_df = df[df['Continent'].isin(selected_continents)]

    fig = px.sunburst(
        filtered_df,
        path=['Continent', 'Country', 'Exchange', 'Index Name'],
        values='count',
        color='Continent',
        color_discrete_map={
            '(?)':'#DDDDDD', 'Global':'#CCCCCC', 'North America': '#1f77b4',
            'Europe': '#ff7f0e', 'Asia': '#2ca02c', 'South America': '#d62728',
            'Africa': '#9467bd', 'Oceania': '#8c564b'
        }
        # We no longer need hover_data because we are removing the ticker from the tooltip.
    )

    fig.update_layout(
        title_text="Geographic Distribution of Global Stock Indices",
        margin=dict(t=50, l=25, r=25, b=25)
    )

    # <<< THE FIX FOR THE TOOLTIP IS HERE >>>
    # This new template is simple and works for ALL layers of the chart.
    fig.update_traces(
        hovertemplate=(
            '<b>%{label}</b><br>' +
            'Indices in this Category: %{value}' +
            '<extra></extra>' # Hides the secondary hover box
        )
    )

    return fig

# --- (Run the App section is unchanged) ---
if __name__ == '__main__':
    app.run(debug=True)